# 01 — Cobertura do repositório IPEA (Fase 1)

Lê `data/interim/metadados.parquet` produzido por `python -m src.scraping`
e descreve o corpus disponível para decidir o corpus-alvo da Fase 2
(briefing §5, Fase 2, decisão inicial).

Critérios de sucesso (briefing §5, Fase 1): N total, N por tipo, min/max de ano.

In [ ]:
from pathlib import Path

import pandas as pd

PARQUET = Path("../data/interim/metadados.parquet")
assert PARQUET.exists(), (
    f"{PARQUET} não existe. Rode antes:\n"
    "    uv run python -m src.scraping --limit 2\n"
    "(ou sem --limit para o corpus completo)."
)
df = pd.read_parquet(PARQUET)
print(f"Total: {len(df):,} documentos")
df.head(3)

## Cobertura temporal

In [ ]:
anos = df["ano"].dropna().astype(int)
print(f"Ano mínimo: {anos.min()}")
print(f"Ano máximo: {anos.max()}")
print(f"Sem ano: {df['ano'].isna().sum()} ({df['ano'].isna().mean():.1%})")
ax = anos.value_counts().sort_index().plot(
    kind="bar", figsize=(14, 4), title="Documentos por ano"
)
ax.set_xlabel("Ano")
ax.set_ylabel("N");

## Distribuição por tipo documental

In [ ]:
tipos = df["tipo"].fillna("(sem tipo)").value_counts()
print(tipos.head(20).to_string())

## Cobertura de handle e resumo

Documentos sem `handle` não podem ter PDF baixado na Fase 2.

In [ ]:
print(f"Com handle:  {df['handle'].notna().sum():>6} ({df['handle'].notna().mean():.1%})")
print(f"Com resumo:  {(df['resumo'] != '').sum():>6} ({(df['resumo'] != '').mean():.1%})")
print(f"Com autores: {(df['autores'] != '').sum():>6} ({(df['autores'] != '').mean():.1%})")

## Top autores e palavras-chave

In [ ]:
autores = (
    df["autores"].fillna("").str.split("; ").explode().str.strip()
)
autores = autores[autores != ""]
print("Top 30 autores:")
print(autores.value_counts().head(30).to_string())

In [ ]:
palavras = (
    df["palavras_chave"].fillna("").str.split(", ").explode().str.strip()
)
palavras = palavras[palavras != ""]
print("Top 50 palavras-chave:")
print(palavras.value_counts().head(50).to_string())

## Cruzamento tipo × década

Base para a decisão de filtrar corpus-alvo na Fase 2.

In [ ]:
df_com_ano = df.dropna(subset=["ano"]).copy()
df_com_ano["decada"] = (df_com_ano["ano"].astype(int) // 10) * 10
pivot = (
    df_com_ano.groupby(["decada", "tipo"]).size().unstack(fill_value=0)
)
pivot = pivot.loc[:, pivot.sum().sort_values(ascending=False).head(8).index]
pivot